# Transformer Encoder 文本分类

## 学习目标

实现并训练一个使用 embedding、位置编码、Multi-Head Attention 和 Transformer Encoder 的分类模型。所有序列张量使用 `(batch, sequence, embedding)`，模块显式设置 `batch_first=True`。

## 概念模型

Self-Attention 先计算 `Q @ K.transpose(-2, -1) / sqrt(d_k)`，再应用 mask 和 softmax，最后加权汇总 V。padding mask 忽略补齐 token，causal mask 阻止看到未来 token。

In [ ]:
import math
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch, seq, embed = 2, 5, 8
tokens = torch.randint(0, 20, (batch, seq))
padding_mask = torch.zeros(batch, seq, dtype=torch.bool)
padding_mask[1, -2:] = True
causal_mask = torch.ones(seq, seq, dtype=torch.bool).tril()
print(tokens.shape, padding_mask.shape, causal_mask.shape, device)

### 实验 1：Attention 的 shape 和 mask

**实验目的**：从 Q/K/V 直接计算 causal attention。Q、K 为 `(batch,seq,embed)`，分数为 `(batch,seq,seq)`；V 最后一维为 6，所以输出为 `(batch,seq,6)`。

mask 在 softmax 前把未来位置填为 `-inf`，相应概率为 0。softmax 沿 key 维执行，每个 query 行权重和应为 1；除以 `sqrt(embed)` 控制高维点积方差。


In [ ]:
q = torch.randn(batch, seq, embed)
k = torch.randn(batch, seq, embed)
v = torch.randn(batch, seq, 6)
scores = q @ k.transpose(-2, -1) / math.sqrt(embed)
scores = scores.masked_fill(~causal_mask, float('-inf'))
weights = scores.softmax(dim=-1)
output = weights @ v
assert output.shape == (batch, seq, 6)
assert torch.allclose(weights.triu(diagonal=1), torch.zeros_like(weights.triu(diagonal=1)))
print('scores', scores.shape, 'weights', weights.shape, 'output', output.shape)

### 实验 2：Embedding、位置编码和 Encoder

**实验目的**：构建最小文本分类器。Embedding 把 long token id 映射为向量；可学习位置参数补充顺序信息；Encoder 包含多头注意力、残差、LayerNorm 和前馈网络。

必须保证序列长度不超过 `max_len`、width 可被 heads 整除、token id 未越界。使用 padding 时，pooling 也应排除补齐位置。


In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab=20, width=16, heads=2, layers=1, classes=2, max_len=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab, width)
        self.position = nn.Parameter(torch.zeros(1, max_len, width))
        layer = nn.TransformerEncoderLayer(d_model=width, nhead=heads, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.head = nn.Linear(width, classes)
    def forward(self, token_ids, padding_mask=None):
        h = self.embedding(token_ids) + self.position[:, :token_ids.size(1)]
        h = self.encoder(h, src_key_padding_mask=padding_mask)
        if padding_mask is None:
            pooled = h.mean(dim=1)
        else:
            valid = (~padding_mask).unsqueeze(-1)
            pooled = (h * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
        return self.head(pooled)

model = TextClassifier().to(device)
logits = model(tokens.to(device), padding_mask.to(device))
assert logits.shape == (batch, 2)
print('logits:', logits.shape)

### 实验 3：短训练和推理

**实验目的**：在 token 总和奇偶分类任务上验证数据流和梯度连通。每个 batch 执行清梯度、前向、交叉熵、反向和 Adam 更新，随后在 eval/inference 模式检查输出。

短训练只属于冒烟测试，不代表模型已稳定学会规律；应使用独立验证集判断泛化。


In [ ]:
data = torch.randint(0, 20, (64, 6))
labels = (data.sum(dim=1) % 2).long()
loader = DataLoader(TensorDataset(data, labels), batch_size=16, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(2):
    model.train()
    for x, y in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(x.to(device)), y.to(device))
        loss.backward()
        optimizer.step()
model.eval()
with torch.inference_mode():
    prediction = model(data[:3].to(device)).argmax(dim=1)
print('inference:', prediction.tolist())

## 官方教程补充

**对应官方源文件：** `beginner_source/transformer_tutorial.rst`、`intermediate_source/transformer_building_blocks.py`、`intermediate_source/variable_length_attention_tutorial.py`

官方 Transformer 材料把 embedding、位置信息、多头自注意力、残差/LayerNorm 和前馈层组成 encoder block。padding mask 表示哪些 token 无效，causal mask 表示时间可见性，两者不可混用；聚合序列时也要排除 padding。现代 attention 内核对 batch-first 和嵌套/变长表示有优化，但模型的 logits、mask 与长度契约应先独立测试。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

写出 `tokens -> embedding -> encoder -> pooling -> logits` 的 shape。说明为什么 padding mask 的 shape 是 `(batch, sequence)`，而 causal mask 的 shape 是 `(sequence, sequence)`。

## 试一试

把 `src_key_padding_mask` 方向写反并观察结果；把 `batch_first` 改成默认值后记录报错并修复。

## 常见错误与调试

检查 token id 是否越界、embedding 宽度是否能被 head 数整除、mask 是否为 bool、pooling 是否排除了 padding，以及所有输入是否位于同一 device。